# 00 — Normativa Italiana: Appalti Pubblici

Estrae il dataset degli atti italiani sugli appalti da **Normattiva**, compatibile
con la pipeline del progetto (notebook 02→04).

## Strategia

```
Per ogni atto del catalogo seed:
  1. HTML Normattiva (AKN) → articoli strutturati, testo CONSOLIDATO vigente
  2. Fallback: PDF Gazzetta Ufficiale → slicing per URN target → articoli
  3. Validazione bloccante: heading/titolo del risultato = atteso?
```

L'HTML AKN è la sorgente primaria perché è legato all'URN dell'atto e contiene
solo quell'atto (niente contaminazione da fascicoli GU con più atti). Il PDF GU
è fallback per atti dove AKN non è popolato (tipicamente atti abrogati o molto
vecchi).

## Output

```
data/output/appalti_it/
├── nodes_it.csv           ← metadati (compatibile con nodes_focal.csv)
├── nodes_texts_it.csv     ← + full_text per notebook 03/04
├── edges_it.csv           ← citazioni estratte
├── edges_it_internal.csv  ← solo citazioni seed→seed
├── validation_report.csv
├── quality_report.md
└── raw/                   ← HTML e PDF scaricati (cache)
```

## Principi implementativi

- **Una funzione per cella, esecuzione in cella separata.** Niente `def` + loop
  nella stessa cella (era la causa dei patch silenziosi della versione
  precedente).
- **Nessuna cache dei risultati parsed.** Si cachano solo i download HTTP
  (HTML e PDF). Il parsing gira sempre da zero — costa secondi ed elimina bug
  da stato sporco.
- **Validazione prima di esportare.** Se anche un solo atto fallisce il check,
  il CSV non viene scritto. Blocco esplicito.


## 1. Setup

In [1]:
import re
import time
import json
import requests
import pandas as pd
import fitz            # pymupdf
import pdfplumber
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import date
from IPython.display import display

OUT_DIR = Path('..') / 'data' / 'output' / 'appalti_it'
RAW_DIR = OUT_DIR / 'raw'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── HTTP config ─────────────────────────────────────────────────────────────
DELAY       = 2.0
TIMEOUT     = 45
MAX_RETRIES = 3
HEADERS     = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0.0.0 Safari/537.36'),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'it-IT,it;q=0.9,en-US;q=0.7,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}
NORMATTIVA_N2LS = 'https://www.normattiva.it/uri-res/N2Ls'

# ── Parametri pipeline ──────────────────────────────────────────────────────
MIN_ARTICOLI_AKN = 1        # soglia per accettare l'AKN
DELAY_ARTICOLO = 1.0        # attesa tra richieste di singoli articoli (vedi notebook V2 notes)
HALT_ON_VALIDATION_FAIL = True

print(f'Output:   {OUT_DIR.resolve()}')
print(f'Cache:    {RAW_DIR.resolve()}')
print('Setup completato.')

Output:   C:\Users\claud\Documents\GitHub\eu-law-network-viz\data\output\appalti_it
Cache:    C:\Users\claud\Documents\GitHub\eu-law-network-viz\data\output\appalti_it\raw
Setup completato.


## 2. Catalogo Seed

29 atti selezionati per coprire tutti i layer normativi attesi:
codici, correttivi, leggi delega, regolamenti attuativi, ibridi PNRR,
leggi trasversali (anticorruzione, trasparenza, antimafia), direttiva servizi.

In [2]:
ATTI_SEED = [

    # ── CODICI ────────────────────────────────────────────────────────────────
    {'slug': 'dlgs_36_2023',  'label': 'D.Lgs. 36/2023',
     'tipo': 'decreto.legislativo', 'data': '2023-03-31', 'numero': '36',
     'titolo': 'Codice dei contratti pubblici',
     'urn': 'urn:nir:stato:decreto.legislativo:2023-03-31;36',
     'layer_atteso': 'Codice primario',
     'note': 'Codice vigente — recepisce Dir. 2014/23, 2014/24, 2014/25',
     'eu_celex_collegati': ['32014L0023', '32014L0024', '32014L0025']},

    {'slug': 'dlgs_50_2016',  'label': 'D.Lgs. 50/2016',
     'tipo': 'decreto.legislativo', 'data': '2016-04-18', 'numero': '50',
     'titolo': 'Codice dei contratti pubblici (abrogato)',
     'urn': 'urn:nir:stato:decreto.legislativo:2016-04-18;50',
     'layer_atteso': 'Codice primario — storico',
     'note': 'Vecchio codice — contratti in corso e analisi storica',
     'eu_celex_collegati': ['32014L0023', '32014L0024', '32014L0025']},

    {'slug': 'dlgs_163_2006', 'label': 'D.Lgs. 163/2006',
     'tipo': 'decreto.legislativo', 'data': '2006-04-12', 'numero': '163',
     'titolo': 'Codice De Lise',
     'urn': 'urn:nir:stato:decreto.legislativo:2006-04-12;163',
     'layer_atteso': 'Codice primario — storico',
     'note': 'Primo codice organico — analisi stratificazione normativa',
     'eu_celex_collegati': ['32004L0017', '32004L0018']},

    # ── CORRETTIVI ────────────────────────────────────────────────────────────
    {'slug': 'dlgs_209_2024', 'label': 'D.Lgs. 209/2024',
     'tipo': 'decreto.legislativo', 'data': '2024-12-31', 'numero': '209',
     'titolo': 'Correttivo al Codice dei contratti pubblici',
     'urn': 'urn:nir:stato:decreto.legislativo:2024-12-31;209',
     'layer_atteso': 'Correttivo',
     'note': 'Correttivo al D.Lgs. 36/2023 — dicembre 2024',
     'eu_celex_collegati': []},

    {'slug': 'dlgs_56_2017',  'label': 'D.Lgs. 56/2017',
     'tipo': 'decreto.legislativo', 'data': '2017-04-19', 'numero': '56',
     'titolo': 'Correttivo al D.Lgs. 50/2016',
     'urn': 'urn:nir:stato:decreto.legislativo:2017-04-19;56',
     'layer_atteso': 'Correttivo',
     'note': 'Primo correttivo al vecchio codice',
     'eu_celex_collegati': []},

    # ── LEGGI DELEGA ──────────────────────────────────────────────────────────
    {'slug': 'l_78_2022',     'label': 'L. 78/2022',
     'tipo': 'legge', 'data': '2022-06-21', 'numero': '78',
     'titolo': 'Delega al Governo in materia di contratti pubblici',
     'urn': 'urn:nir:stato:legge:2022-06-21;78',
     'layer_atteso': 'Legge delega',
     'note': 'Delega che ha abilitato il D.Lgs. 36/2023',
     'eu_celex_collegati': []},

    {'slug': 'l_11_2016',     'label': 'L. 11/2016',
     'tipo': 'legge', 'data': '2016-01-28', 'numero': '11',
     'titolo': 'Delega per recepimento Dir. 2014/23, 2014/24, 2014/25',
     'urn': 'urn:nir:stato:legge:2016-01-28;11',
     'layer_atteso': 'Legge delega',
     'note': 'Delega che ha abilitato il D.Lgs. 50/2016',
     'eu_celex_collegati': []},

    # ── REGOLAMENTO ATTUATIVO ─────────────────────────────────────────────────
    {'slug': 'dpr_207_2010',  'label': 'D.P.R. 207/2010',
     'tipo': 'decreto.del.presidente.della.repubblica',
     'data': '2010-10-05', 'numero': '207',
     'titolo': 'Regolamento di esecuzione del D.Lgs. 163/2006',
     'urn': 'urn:nir:stato:decreto.del.presidente.della.repubblica:2010-10-05;207',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Archetipo livello III: puro dettaglio operativo, zero principi',
     'eu_celex_collegati': []},

    # ── CANDIDATI IBRIDITÀ ALTA ───────────────────────────────────────────────
    {'slug': 'l_108_2021',    'label': 'L. 108/2021',
     'tipo': 'legge', 'data': '2021-07-29', 'numero': '108',
     'titolo': 'Governance PNRR e semplificazione (conv. D.L. 77/2021)',
     'urn': 'urn:nir:stato:legge:2021-07-29;108',
     'layer_atteso': 'Ibrido atteso',
     'note': 'PNRR: principi e soglie numeriche operative nello stesso articolo',
     'eu_celex_collegati': []},

    {'slug': 'l_120_2020',    'label': 'L. 120/2020',
     'tipo': 'legge', 'data': '2020-09-11', 'numero': '120',
     'titolo': 'Semplificazione e innovazione digitale (conv. D.L. 76/2020)',
     'urn': 'urn:nir:stato:legge:2020-09-11;120',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Misure urgenti semplificazione appalti: mescola soglie, procedure e principi',
     'eu_celex_collegati': []},

    {'slug': 'l_55_2019',     'label': 'L. 55/2019',
     'tipo': 'legge', 'data': '2019-06-14', 'numero': '55',
     'titolo': 'Sblocca Cantieri (conv. D.L. 32/2019)',
     'urn': 'urn:nir:stato:legge:2019-06-14;55',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Deroghe temporanee al codice mescolate con norme di principio',
     'eu_celex_collegati': []},

    {'slug': 'l_114_2014',    'label': 'L. 114/2014',
     'tipo': 'legge', 'data': '2014-08-11', 'numero': '114',
     'titolo': 'Semplificazione e trasparenza amministrativa (conv. D.L. 90/2014)',
     'urn': 'urn:nir:stato:legge:2014-08-11;114',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Modifica il codice appalti su più livelli nello stesso testo',
     'eu_celex_collegati': []},

    # ── LEGGI TRASVERSALI E ANTICORRUZIONE ────────────────────────────────────
    {'slug': 'l_241_1990',    'label': 'L. 241/1990',
     'tipo': 'legge', 'data': '1990-08-07', 'numero': '241',
     'titolo': 'Legge sul procedimento amministrativo',
     'urn': 'urn:nir:stato:legge:1990-08-07;241',
     'layer_atteso': 'Principi generali',
     'note': 'Richiamata in quasi ogni articolo dei codici appalti',
     'eu_celex_collegati': []},

    {'slug': 'l_190_2012',    'label': 'L. 190/2012',
     'tipo': 'legge', 'data': '2012-11-06', 'numero': '190',
     'titolo': 'Legge anticorruzione',
     'urn': 'urn:nir:stato:legge:2012-11-06;190',
     'layer_atteso': 'Legge trasversale',
     'note': 'Obblighi di trasparenza anticorruzione negli appalti',
     'eu_celex_collegati': []},

    {'slug': 'l_90_2024',     'label': 'L. 90/2024',
     'tipo': 'legge', 'data': '2024-06-28', 'numero': '90',
     'titolo': 'Disposizioni in materia di cybersicurezza',
     'urn': 'urn:nir:stato:legge:2024-06-28;90',
     'layer_atteso': 'Legge trasversale',
     'note': 'Obblighi cybersicurezza per appalti ICT e infrastrutture critiche',
     'eu_celex_collegati': []},

    {'slug': 'l_136_2010',    'label': 'L. 136/2010',
     'tipo': 'legge', 'data': '2010-08-13', 'numero': '136',
     'titolo': 'Piano straordinario contro le mafie — tracciabilità finanziaria appalti',
     'urn': 'urn:nir:stato:legge:2010-08-13;136',
     'layer_atteso': 'Legge trasversale',
     'note': 'Introduce obbligo tracciabilità flussi finanziari negli appalti pubblici',
     'eu_celex_collegati': []},

    # ── TRASPARENZA PA ────────────────────────────────────────────────────────
    {'slug': 'dlgs_33_2013',  'label': 'D.Lgs. 33/2013',
     'tipo': 'decreto.legislativo', 'data': '2013-03-14', 'numero': '33',
     'titolo': 'Riordino degli obblighi di pubblicità, trasparenza e diffusione di informazioni',
     'urn': 'urn:nir:stato:decreto.legislativo:2013-03-14;33',
     'layer_atteso': 'Legge trasversale',
     'note': 'Disciplina la pubblicazione degli atti di gara e dei contratti pubblici',
     'eu_celex_collegati': []},

    {'slug': 'dlgs_97_2016',  'label': 'D.Lgs. 97/2016',
     'tipo': 'decreto.legislativo', 'data': '2016-05-25', 'numero': '97',
     'titolo': 'Revisione e semplificazione disposizioni trasparenza (riforma Madia)',
     'urn': 'urn:nir:stato:decreto.legislativo:2016-05-25;97',
     'layer_atteso': 'Correttivo',
     'note': 'Correttivo di D.Lgs. 33/2013 e L. 190/2012 — riforma Madia',
     'eu_celex_collegati': []},

    # ── ANTIMAFIA ─────────────────────────────────────────────────────────────
    {'slug': 'dlgs_218_2012', 'label': 'D.Lgs. 218/2012',
     'tipo': 'decreto.legislativo', 'data': '2012-11-15', 'numero': '218',
     'titolo': 'Disposizioni integrative e correttive al codice antimafia',
     'urn': 'urn:nir:stato:decreto.legislativo:2012-11-15;218',
     'layer_atteso': 'Correttivo',
     'note': 'Aggiorna la disciplina della documentazione antimafia negli appalti',
     'eu_celex_collegati': []},

    # ── OPERE PUBBLICHE — MONITORAGGIO E VALUTAZIONE ──────────────────────────
    {'slug': 'dlgs_228_2011', 'label': 'D.Lgs. 228/2011',
     'tipo': 'decreto.legislativo', 'data': '2011-12-29', 'numero': '228',
     'titolo': 'Valutazione degli investimenti relativi ad opere pubbliche',
     'urn': 'urn:nir:stato:decreto.legislativo:2011-12-29;228',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Introduce procedure di valutazione ex ante dei grandi appalti pubblici',
     'eu_celex_collegati': []},

    {'slug': 'dlgs_229_2011', 'label': 'D.Lgs. 229/2011',
     'tipo': 'decreto.legislativo', 'data': '2011-12-29', 'numero': '229',
     'titolo': 'Monitoraggio sullo stato di attuazione delle opere pubbliche',
     'urn': 'urn:nir:stato:decreto.legislativo:2011-12-29;229',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Sistema di monitoraggio fisico-finanziario dei contratti pubblici',
     'eu_celex_collegati': []},

    # ── DIRETTIVA SERVIZI ─────────────────────────────────────────────────────
    {'slug': 'dlgs_59_2010',  'label': 'D.Lgs. 59/2010',
     'tipo': 'decreto.legislativo', 'data': '2010-03-26', 'numero': '59',
     'titolo': 'Attuazione Direttiva Servizi 2006/123/CE',
     'urn': 'urn:nir:stato:decreto.legislativo:2010-03-26;59',
     'layer_atteso': 'Codice primario — storico',
     'note': 'Recepisce la Direttiva Bolkestein — rilevante per concessioni di servizi',
     'eu_celex_collegati': ['32006L0123']},

    # ── APPALTI BENI CULTURALI ────────────────────────────────────────────────
    {'slug': 'dm_154_2017',   'label': 'D.M. 154/2017',
     'tipo': 'decreto.ministeriale', 'data': '2017-08-22', 'numero': '154',
     'titolo': 'Regolamento appalti pubblici di lavori riguardanti beni culturali',
     'urn': 'urn:nir:stato:decreto.ministeriale:2017-08-22;154',
     'layer_atteso': 'Regolamento attuativo',
     'note': 'Archetipo livello III di settore — appalti beni culturali tutelati',
     'eu_celex_collegati': []},

    # ── IBRIDI PNRR (aggiuntivi) ──────────────────────────────────────────────
    {'slug': 'dl_13_2023',    'label': 'D.L. 13/2023',
     'tipo': 'decreto.legge', 'data': '2023-02-24', 'numero': '13',
     'titolo': 'Disposizioni urgenti per PNRR e PNC',
     'urn': 'urn:nir:stato:decreto.legge:2023-02-24;13',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Modifica il Codice con deroghe temporanee PNRR — alta ibridità attesa',
     'eu_celex_collegati': []},

    {'slug': 'l_56_2024',     'label': 'L. 56/2024',
     'tipo': 'legge', 'data': '2024-04-29', 'numero': '56',
     'titolo': 'Conversione D.L. 19/2024 — attuazione PNRR',
     'urn': 'urn:nir:stato:legge:2024-04-29;56',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Ulteriori disposizioni urgenti PNRR — mescola principi e tecnicismi',
     'eu_celex_collegati': []},

    {'slug': 'l_105_2025',    'label': 'L. 105/2025',
     'tipo': 'legge', 'data': '2025-07-18', 'numero': '105',
     'titolo': 'Conversione D.L. 73/2025 — infrastrutture strategiche e PNRR',
     'urn': 'urn:nir:stato:legge:2025-07-18;105',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Recente: infrastrutture strategiche + trasporti + PNRR',
     'eu_celex_collegati': []},

    # ── MODIFICHE AL CODICE VIA LEGGI DI BILANCIO / SEMPLIFICAZIONI ───────────
    {'slug': 'l_145_2018',    'label': 'L. 145/2018',
     'tipo': 'legge', 'data': '2018-12-30', 'numero': '145',
     'titolo': 'Legge di bilancio 2019 — modifiche al Codice appalti',
     'urn': 'urn:nir:stato:legge:2018-12-30;145',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Modifica il Codice dentro la legge di bilancio — archetipo di ibridità',
     'eu_celex_collegati': []},

    {'slug': 'dl_135_2018',   'label': 'D.L. 135/2018',
     'tipo': 'decreto.legge', 'data': '2018-12-14', 'numero': '135',
     'titolo': 'Decreto Semplificazioni 2018',
     'urn': 'urn:nir:stato:decreto.legge:2018-12-14;135',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Semplificazioni appalti mescolate con altre materie',
     'eu_celex_collegati': []},

    # ── DURC / ANTICIPO / MISURE URGENTI ──────────────────────────────────────
    {'slug': 'l_98_2013',     'label': 'L. 98/2013',
     'tipo': 'legge', 'data': '2013-08-09', 'numero': '98',
     'titolo': 'Conversione D.L. 69/2013 — Disposizioni urgenti rilancio economia',
     'urn': 'urn:nir:stato:legge:2013-08-09;98',
     'layer_atteso': 'Ibrido atteso',
     'note': 'Introduce anticipo 10% e DURC esteso — interviene sul Codice 163/2006',
     'eu_celex_collegati': []},

]

# URL Normattiva per ogni atto
for atto in ATTI_SEED:
    base = NORMATTIVA_N2LS + '?' + atto['urn']
    atto['html_url'] = base

df_seed = pd.DataFrame(ATTI_SEED)
print(f'Atti nel catalogo seed: {len(ATTI_SEED)}')
print()
for layer, grp in df_seed.groupby('layer_atteso'):
    print(f'  {layer}')
    print(f'    → {", ".join(grp["label"].tolist())}')

Atti nel catalogo seed: 29

  Codice primario
    → D.Lgs. 36/2023
  Codice primario — storico
    → D.Lgs. 50/2016, D.Lgs. 163/2006, D.Lgs. 59/2010
  Correttivo
    → D.Lgs. 209/2024, D.Lgs. 56/2017, D.Lgs. 97/2016, D.Lgs. 218/2012
  Ibrido atteso
    → L. 108/2021, L. 120/2020, L. 55/2019, L. 114/2014, D.L. 13/2023, L. 56/2024, L. 105/2025, L. 145/2018, D.L. 135/2018, L. 98/2013
  Legge delega
    → L. 78/2022, L. 11/2016
  Legge trasversale
    → L. 190/2012, L. 90/2024, L. 136/2010, D.Lgs. 33/2013
  Principi generali
    → L. 241/1990
  Regolamento attuativo
    → D.P.R. 207/2010, D.Lgs. 228/2011, D.Lgs. 229/2011, D.M. 154/2017


## 3. HTTP + Cache

Funzioni **pure** per scaricare e cachare HTML e PDF. Zero logica di business:
una funzione scarica, un'altra legge la cache, entrambe ritornano solo bytes/string.

In [3]:
# ── Sessione HTTP globale ───────────────────────────────────────────────────
# Normattiva è un'app stateful: le chiamate a /atto/caricaArticolo richiedono
# un JSESSIONID cookie che viene settato solo visitando la pagina dell'atto
# (/uri-res/N2Ls?urn:...). Se la cache HTML esiste, dobbiamo comunque fare un
# "warm-up" hit per ottenere i cookies.

SESSION = requests.Session()
SESSION.headers.update(HEADERS)
_warmed_up_atti = set()        # slug degli atti già riscaldati in questa run


def safe_get(url, retries=MAX_RETRIES, extra_headers=None):
    """GET con retry esponenziale, usa la SESSION per preservare i cookies."""
    hdrs = dict(SESSION.headers)
    if extra_headers:
        hdrs.update(extra_headers)
    for attempt in range(retries):
        try:
            r = SESSION.get(url, headers=hdrs, timeout=TIMEOUT, allow_redirects=True)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            wait = DELAY * (2 ** attempt)
            print(f'    retry {attempt+1}/{retries}: {e} — attendo {wait:.0f}s')
            time.sleep(wait)
    return None


def warm_up_session(atto):
    """Fa un GET alla pagina padre dell'atto per ottenere JSESSIONID.
    No-op se già fatto in questa run per questo atto."""
    slug = atto['slug']
    if slug in _warmed_up_atti:
        return True
    resp = safe_get(atto['html_url'])
    time.sleep(DELAY)
    if resp and resp.status_code == 200:
        _warmed_up_atti.add(slug)
        return True
    return False


def fetch_html_normattiva(atto):
    """Scarica HTML Normattiva. Ritorna (html_text, 'cache'|'fresh'|None).
    NOTA: anche leggendo da cache, fa warm-up di sessione."""
    slug = atto['slug']
    html_path = RAW_DIR / f'{slug}.html'

    if html_path.exists():
        # Cache hit — ma dobbiamo comunque riscaldare la sessione per le chiamate
        # successive a caricaArticolo
        warm_up_session(atto)
        return html_path.read_text(encoding='utf-8', errors='replace'), 'cache'

    # Cache miss — la stessa GET che scarica l'HTML stabilisce anche la sessione
    resp = safe_get(atto['html_url'])
    time.sleep(DELAY)
    if not resp or resp.status_code != 200 or len(resp.text) < 5000:
        return None, None
    html_path.write_text(resp.text, encoding='utf-8')
    _warmed_up_atti.add(slug)
    return resp.text, 'fresh'


def fetch_gu_pdf(gu_pdf_url, slug):
    """Scarica PDF Gazzetta Ufficiale. Ritorna (Path, 'cache'|'fresh'|None)."""
    for candidate in [RAW_DIR / f'{slug}_gu.pdf', RAW_DIR / f'{slug}.pdf']:
        if candidate.exists():
            return candidate, 'cache'
    if not gu_pdf_url:
        return None, None
    resp = safe_get(gu_pdf_url)
    time.sleep(DELAY)
    if not resp or resp.status_code != 200:
        return None, None
    ct = resp.headers.get('Content-Type', '')
    if 'pdf' in ct or resp.content[:4] == b'%PDF':
        pdf_path = RAW_DIR / f'{slug}_gu.pdf'
        pdf_path.write_bytes(resp.content)
        return pdf_path, 'fresh'
    return None, None


# ── Fetch singolo articolo (endpoint /atto/caricaArticolo) ─────────────────

NORMATTIVA_ARTICOLO_URL = 'https://www.normattiva.it/atto/caricaArticolo'


def _post_caricaArticolo(params, referer):
    """Fallback POST a /atto/caricaArticolo con form-data (usa SESSION)."""
    hdrs = dict(SESSION.headers)
    hdrs.update({
        'Referer': referer,
        'X-Requested-With': 'XMLHttpRequest',
        'Origin': 'https://www.normattiva.it',
    })
    for attempt in range(MAX_RETRIES):
        try:
            r = SESSION.post(NORMATTIVA_ARTICOLO_URL, data=params,
                             headers=hdrs, timeout=TIMEOUT, allow_redirects=True)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            wait = DELAY * (2 ** attempt)
            print(f'    POST retry {attempt+1}/{MAX_RETRIES}: {e} — attendo {wait:.0f}s')
            time.sleep(wait)
    return None


def fetch_articolo(slug, query_string, id_articolo, referer_url):
    """Scarica un singolo articolo.

    Args:
        slug:           slug dell'atto padre (per naming cache)
        query_string:   QS letta dall'onclick showArticle (es. 'art.versione=5&...')
        id_articolo:    ID numerico dell'articolo
        referer_url:    URL della pagina padre (usata come Referer header)
    """
    art_path = RAW_DIR / f'{slug}_art_{int(id_articolo):03d}.html'
    if art_path.exists():
        cached = art_path.read_text(encoding='utf-8', errors='replace')
        if len(cached) >= 100:
            return cached, 'cache'

    url = f'{NORMATTIVA_ARTICOLO_URL}?{query_string}'
    extra = {
        'Referer': referer_url,
        'X-Requested-With': 'XMLHttpRequest',
    }
    # 1) GET
    resp = safe_get(url, extra_headers=extra)
    time.sleep(DELAY_ARTICOLO)

    got_it = resp and resp.status_code == 200 and len(resp.text) >= 100
    method = 'fresh_get'

    if not got_it:
        # 2) fallback POST
        from urllib.parse import parse_qsl
        params = dict(parse_qsl(query_string))
        resp = _post_caricaArticolo(params, referer_url)
        time.sleep(DELAY_ARTICOLO)
        got_it = resp and resp.status_code == 200 and len(resp.text) >= 100
        method = 'fresh_post'

    if not got_it:
        return None, None

    art_path.write_text(resp.text, encoding='utf-8')
    return resp.text, method


print('✓ HTTP helpers + SESSION + warm_up_session + fetch_articolo (GET+POST, con Referer)')

✓ HTTP helpers + SESSION + warm_up_session + fetch_articolo (GET+POST, con Referer)


## 4. Primitive di Parsing

Funzioni pure per:
- trovare intestazioni ufficiali (`DECRETO LEGISLATIVO 12 aprile 2006, n. 163`)
- estrarre articoli da testo libero via regex
- estrarre citazioni URN da testo
- parsare HTML AKN (classi Normattiva)
- slicing di un atto dentro un fascicolo GU multi-atto
- estrarre testo da PDF via pymupdf con filtro intestazioni GU

Il **slicing** è l'elemento chiave per gestire i PDF GU che contengono più atti:
usa l'**ultima** occorrenza dell'heading target come inizio del body (le
occorrenze precedenti sono tipicamente copertina e sommario).

In [4]:
# ── Heading detection ──────────────────────────────────────────────────────

MESI = {'gennaio':1, 'febbraio':2, 'marzo':3, 'aprile':4, 'maggio':5, 'giugno':6,
        'luglio':7, 'agosto':8, 'settembre':9, 'ottobre':10, 'novembre':11, 'dicembre':12}

_HEADING_PATTERNS = [
    ('decreto.legislativo',
     r'DECRETO\s+LEGISLATIVO\s+(\d{1,2})\s+(\w+)\s+(\d{4})\s*,?\s*n\.?\s*(\d+)'),
    ('decreto.del.presidente.della.repubblica',
     r'DECRETO\s+DEL\s+PRESIDENTE\s+DELLA\s+REPUBBLICA\s+(\d{1,2})\s+(\w+)\s+(\d{4})\s*,?\s*n\.?\s*(\d+)'),
    ('decreto.legge',
     r'DECRETO[\s\-]LEGGE\s+(\d{1,2})\s+(\w+)\s+(\d{4})\s*,?\s*n\.?\s*(\d+)'),
    ('legge',
     r'\bLEGGE\s+(\d{1,2})\s+(\w+)\s+(\d{4})\s*,?\s*n\.?\s*(\d+)'),
]


def find_all_headings(text):
    """Tutte le occorrenze di heading ufficiali nel testo. No dedup, sort per posizione.

    Ogni elemento: {tipo, data (YYYY-MM-DD), numero, pos}.
    """
    found = []
    for tipo, pat in _HEADING_PATTERNS:
        for m in re.finditer(pat, text, re.IGNORECASE):
            mese = MESI.get(m.group(2).lower())
            if not mese:
                continue
            # escludi "LEGGE" catturata dentro "DECRETO-LEGGE"/"DECRETO LEGGE"
            if tipo == 'legge':
                pre = text[max(0, m.start() - 15):m.start()].upper()
                if re.search(r'DECRETO[\s\-]*$', pre):
                    continue
            found.append({
                'tipo':   tipo,
                'data':   f'{int(m.group(3)):04d}-{mese:02d}-{int(m.group(1)):02d}',
                'numero': m.group(4),
                'pos':    m.start(),
            })
    found.sort(key=lambda x: x['pos'])
    return found


# ── Slicing di un atto dentro fascicolo multi-atto ─────────────────────────

def slice_atto_from_fascicolo(full_text, atto):
    """Isola il body di `atto` dentro un fascicolo GU multi-atto.

    Ritorna (text_slice, status, info). Strategia: prende l'ultima occorrenza
    dell'heading target come body-start (le precedenti sono copertina/sommario).
    Se non produce articoli, prova le occorrenze precedenti.
    """
    heads = find_all_headings(full_text)
    info = {'n_headings': len(heads), 'n_atti_unici': 0, 'target_occ': 0, 'tried': 0}
    if not heads:
        return full_text, 'no_heading', info

    target_key = (atto['tipo'], atto['data'], atto['numero'])
    target_positions = [i for i, h in enumerate(heads)
                        if (h['tipo'], h['data'], h['numero']) == target_key]
    info['n_atti_unici'] = len({(h['tipo'], h['data'], h['numero']) for h in heads})
    info['target_occ'] = len(target_positions)

    if not target_positions:
        return full_text, 'target_not_found', info

    def _slice_at(idx):
        start = heads[idx]['pos']
        end = len(full_text)
        for h in heads[idx + 1:]:
            if (h['tipo'], h['data'], h['numero']) != target_key:
                end = h['pos']
                break
        return full_text[start:end]

    # prova dall'ULTIMA occorrenza verso le precedenti
    for tried, idx in enumerate(reversed(target_positions), start=1):
        info['tried'] = tried
        sl = _slice_at(idx)
        if estrai_articoli_da_testo(sl):
            status = 'single_ok' if info['n_atti_unici'] == 1 else 'sliced'
            return sl, status, info

    # nessuna occorrenza produce articoli → restituisci comunque l'ultima
    return _slice_at(target_positions[-1]), 'sliced_no_articles', info


# ── Estrazione articoli via regex su testo libero ──────────────────────────

RE_ARTICOLO = re.compile(
    r'(?m)^[ \t]*Art(?:icolo)?\.?\s+'
    r'(\d+(?:\s*-?\s*(?:bis|ter|quater|quinquies|sexies|septies|octies|novies|decies))?)'
    r'[ \t]*\.?[ \t]*(?:\(([^)\n]{0,120})\))?[ \t]*$',
    re.IGNORECASE
)
RE_COMMA_COUNT = re.compile(r'(?m)^\s*\d+\.\s+\S')


def estrai_articoli_da_testo(full_text):
    """Segmenta full_text in articoli tramite heading 'Art. N.' ancorato a inizio riga."""
    splits = list(RE_ARTICOLO.finditer(full_text))
    if len(splits) < 2:
        return []
    articoli = []
    for i, m in enumerate(splits):
        art_num = m.group(1).strip()
        rubrica = (m.group(2) or '').strip()
        start   = m.start()
        end     = splits[i+1].start() if i+1 < len(splits) else len(full_text)
        testo   = full_text[start:end].strip()
        articoli.append({
            'id':       f'art{art_num}',
            'numero':   art_num,
            'rubrica':  rubrica,
            'testo':    testo[:10000],
            'n_commi':  max(len(RE_COMMA_COUNT.findall(testo)), 1),
        })
    return articoli


# ── Estrazione citazioni URN ───────────────────────────────────────────────

_TIPO_MAP_CIT = [
    (re.compile(r'(?i)\bdecreto\s+legislativo\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'), 'decreto.legislativo'),
    (re.compile(r'(?i)\bd\.?\s*lgs\.?\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),        'decreto.legislativo'),
    (re.compile(r'(?i)\bdecreto(?:\s+del)?\s*presidente\s+della\s+repubblica\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'), 'decreto.del.presidente.della.repubblica'),
    (re.compile(r'(?i)\bd\.?\s*p\.?\s*r\.?\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),  'decreto.del.presidente.della.repubblica'),
    (re.compile(r'(?i)\bdecreto[\s\-]legge\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),     'decreto.legge'),
    (re.compile(r'(?i)\bd\.?\s*l\.?\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),            'decreto.legge'),
    (re.compile(r'(?i)\blegge\s+(?:n\.?\s*)?(\d+)\s*/\s*(\d{4})'),                    'legge'),
    (re.compile(r'(?i)\bl\.\s*(\d+)\s*/\s*(\d{4})'),                                    'legge'),
]


def estrai_citazioni_da_testo(testo):
    """Estrae citazioni 'D.Lgs. 163/2006', 'L. 241/1990' ecc. Ritorna lista {urn, testo}."""
    citazioni, seen_urns = [], set()
    for pat, tipo in _TIPO_MAP_CIT:
        for m in pat.finditer(testo):
            try:
                numero, anno = m.group(1).strip(), m.group(2).strip()
            except IndexError:
                continue
            if not anno.isdigit() or not (1948 <= int(anno) <= 2025):
                continue
            urn = f'urn:nir:stato:{tipo}:{anno};{numero}'
            if urn in seen_urns:
                continue
            seen_urns.add(urn)
            citazioni.append({'urn': urn, 'testo': m.group(0).strip()})
    return citazioni


# ── Estrazione testo da PDF ────────────────────────────────────────────────

_GU_SKIP = [
    'Gazzetta Ufficiale', 'Spediz. abb', 'ISTITUTO POLIGRAFICO',
    'SI PUBBLICA TUTTI', 'SOMMARIO', 'CENTRALINO',
    'COPIA TRATTA DA GURITEL', 'Supplemento ordinario',
    'Serie generale - n.', 'LEGGI ED ALTRI ATTI',
]


def estrai_testo_da_pdf(pdf_path):
    """Testo via pymupdf, filtra intestazioni GU, ricongiunge parole spezzate."""
    doc = fitz.open(str(pdf_path))
    pages_text = []
    for page in doc:
        blocks = page.get_text('blocks', sort=True)
        lines = []
        for b in blocks:
            txt = b[4].strip()
            if not txt or len(txt) < 5:
                continue
            if any(p in txt for p in _GU_SKIP):
                continue
            lines.append(txt)
        if lines:
            pages_text.append('\n'.join(lines))
    doc.close()
    full = '\n'.join(pages_text)
    full = re.sub(r'-\n\s*', '', full)   # parole spezzate a fine riga
    return full


# ── Parsing HTML AKN Normattiva ────────────────────────────────────────────

def estrai_meta_da_html(html):
    """Meta + citazioni URN + link al PDF GU originale dalla pagina Normattiva."""
    soup = BeautifulSoup(html, 'html.parser')
    meta = {
        'codice_redazionale': '',
        'data_gu':            '',
        'titolo':             '',
        'preambolo':          '',
        'urn_citazioni':      [],
        'gu_pdf_url':         '',
    }

    for inp in soup.find_all('input', {'name': True}):
        name, val = inp.get('name', ''), inp.get('value', '')
        if 'codiceRedazionale' in name and val:
            meta['codice_redazionale'] = val
        if 'dataPubblicazioneGazzetta' in name and val:
            meta['data_gu'] = val.replace('-', '')

    if not meta['codice_redazionale']:
        for a in soup.find_all('a', href=True):
            m = re.search(r'codiceRedaz=([A-Z0-9]+)', a['href'])
            if m:
                meta['codice_redazionale'] = m.group(1)
            m2 = re.search(r'dataGU=(\d{8})', a['href'])
            if m2:
                meta['data_gu'] = m2.group(1)

    testa = soup.find(class_='testa_atto')
    meta['titolo'] = testa.get_text(separator=' ', strip=True)[:300] if testa else ''

    preamble_parts = []
    for cls in ('preamble-title-akn', 'preamble-citations-akn',
                'preamble-text-akn', 'preamble-end-akn', 'formula-introduttiva'):
        for tag in soup.find_all(class_=cls):
            t = tag.get_text(separator=' ', strip=True)
            if t:
                preamble_parts.append(t)
    meta['preambolo'] = ' '.join(preamble_parts)

    seen = set()
    for a in soup.find_all('a', href=True):
        m = re.search(r'(urn:nir:[^\s\'"<>&]+)', a['href'])
        if m:
            urn_base = re.sub(r'~.*$', '', m.group(1).rstrip('~/ '))
            if urn_base not in seen:
                seen.add(urn_base)
                meta['urn_citazioni'].append({'urn': urn_base, 'testo': a.get_text(strip=True)})

    # Link PDF GU: preferisci quello con dataGU = meta['data_gu'] (pubblicazione originale)
    data_gu_target = meta['data_gu']
    best_url = None
    fallback_url = None
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'gazzettaufficiale.it' not in href or 'pdf' not in href.lower():
            continue
        if fallback_url is None:
            fallback_url = href
        if data_gu_target and data_gu_target in href:
            best_url = href
            break
    meta['gu_pdf_url'] = best_url or fallback_url or ''

    return meta


# ── Parsing nuova strategia: lista articoli dalla navigazione sidebar ──────

from urllib.parse import parse_qsl

def estrai_lista_articoli_da_nav(html):
    """Dalla pagina principale estrae [(id_articolo, qs, numero_testuale), ...]
    prendendo solo la VERSIONE VIGENTE di ogni articolo (primo link per id).

    La sidebar Normattiva ha link con classe `numero_articolo` e
    onclick="return showArticle('/atto/caricaArticolo?ART.PARAMS...', this);"

    Per ogni articolo ci sono più versioni (agg.1, agg.2, orig...): quella
    VIGENTE è la prima che compare in lettura DOM. Quindi prendiamo solo la
    prima occorrenza di ciascun `art.idArticolo`.
    """
    soup = BeautifulSoup(html, 'html.parser')
    seen_id = set()
    out = []
    for a in soup.find_all('a', class_='numero_articolo'):
        onclick = a.get('onclick', '') or ''
        m = re.search(r"showArticle\('([^']+)'", onclick)
        if not m:
            continue
        url_frag = m.group(1)
        # url_frag è tipo "/atto/caricaArticolo?art.versione=5&art.idGruppo=1&..."
        qs = url_frag.split('?', 1)[-1].rstrip('&')
        params = dict(parse_qsl(qs))
        id_art = params.get('art.idArticolo')
        if not id_art:
            continue
        if id_art in seen_id:
            continue
        seen_id.add(id_art)
        numero_testuale = a.get_text(strip=True) or id_art
        try:
            id_art_int = int(id_art)
        except ValueError:
            id_art_int = len(out) + 1
        out.append((id_art_int, qs, numero_testuale))
    out.sort(key=lambda x: x[0])
    return out


def _parse_fragment_articolo(html_fragment):
    """Estrae testo + rubrica dal frammento HTML di un singolo articolo.

    Il frammento restituito da caricaArticolo contiene le classi AKN:
    `article-num-akn`, `article-heading-akn`, `art-commi-div-akn`.
    Se le classi AKN non ci sono, ripieghiamo sul testo completo del frammento.
    """
    soup = BeautifulSoup(html_fragment, 'html.parser')

    # rubrica: classe `article-heading-akn`
    heading_tag = soup.find(class_='article-heading-akn')
    rubrica = heading_tag.get_text(separator=' ', strip=True) if heading_tag else ''

    # corpo: classe `art-commi-div-akn` (tutti i commi). Può essere multipla.
    commi_divs = soup.find_all(class_='art-commi-div-akn')
    if commi_divs:
        corpo_raw = ' '.join(d.get_text(separator=' ', strip=True) for d in commi_divs)
    else:
        # fallback: prendi tutto il testo del frammento, filtrando UI noise
        corpo_raw = soup.get_text(separator=' ', strip=True)
        # rimuovi i numeri di articolo ripetuti dalla sidebar
        corpo_raw = re.sub(r'\b\d+(?:-[a-z]+)?\b(?=\s+\d)', '', corpo_raw, count=0)

    corpo = re.sub(r'\s+', ' ', corpo_raw).strip()
    return rubrica, corpo


def estrai_articoli_multirichiesta(html, slug, referer_url):
    """Nuova pipeline AKN: scarica un articolo alla volta dalla sidebar.

    Ritorna lista di dict nello stesso formato della vecchia
    estrai_articoli_da_html_akn: {id, numero, rubrica, testo, n_commi}.
    """
    lista = estrai_lista_articoli_da_nav(html)
    if not lista:
        return []

    art_segs = []
    n_tot = len(lista)
    n_fail = 0
    for i, (id_art, qs, numero_testuale) in enumerate(lista):
        frag, method = fetch_articolo(slug, qs, id_art, referer_url)
        if not frag:
            n_fail += 1
            continue
        rubrica, testo = _parse_fragment_articolo(frag)
        if len(testo) < 20:
            n_fail += 1
            continue
        art_segs.append({
            'id':       f'art{numero_testuale}',
            'numero':   numero_testuale,
            'rubrica':  rubrica[:500],
            'testo':    f'Art. {numero_testuale}. {rubrica}\n{testo}'[:15000],
            'n_commi':  max(1, len(re.findall(r'(?m)^\s*\d+\.\s+\S', testo))),
        })
    # log interno per diagnostica, senza inquinare output
    if n_fail:
        print(f'    [{slug}] articoli: {len(art_segs)}/{n_tot} estratti, {n_fail} falliti')
    return art_segs


def estrai_articoli_da_html_akn(html, slug=None, referer_url=None):
    """Deprecata. Usa estrai_articoli_multirichiesta(html, slug, referer_url)."""
    if slug is None or referer_url is None:
        return []
    return estrai_articoli_multirichiesta(html, slug, referer_url)


print('✓ Primitive di parsing definite')

✓ Primitive di parsing definite


## 5. Pipeline per Singolo Atto

`process_atto(atto)` è la funzione che orchestra tutto per un singolo atto.
Strategia: **AKN → PDF GU sliced → fallimento**. Ritorna un dict con struttura
uniforme, che diventerà una riga dei CSV finali.

In [5]:
def _empty_result(slug, label, errore):
    return {
        'slug': slug, 'label': label,
        'fonte': 'nessuna', 'fonte_file': None,
        'titolo_estratto': '', 'preambolo': '',
        'articoli': [], 'citazioni': [],
        'n_articoli': 0, 'n_citazioni': 0, 'n_commi': 0,
        'full_text': '',
        'slice_info': None,
        'errore': errore,
    }


def _finalize_from_akn(slug, label, meta, art_segs, atto):
    full_text = '\n\n'.join(a['testo'] for a in art_segs)
    # Le citazioni arrivano dagli href <a href='urn:nir:...'> del HTML.
    # Aggiungiamo anche citazioni testuali per robustezza.
    cit_testuali = estrai_citazioni_da_testo(full_text)
    seen = {c['urn'] for c in meta['urn_citazioni']}
    all_cit = list(meta['urn_citazioni'])
    for c in cit_testuali:
        if c['urn'] not in seen:
            seen.add(c['urn'])
            all_cit.append(c)
    return {
        'slug': slug, 'label': label,
        'fonte': 'html_akn', 'fonte_file': f'{slug}.html',
        'titolo_estratto': meta['titolo'] or atto['titolo'],
        'preambolo': meta['preambolo'],
        'articoli': art_segs,
        'citazioni': all_cit,
        'n_articoli': len(art_segs),
        'n_citazioni': len(all_cit),
        'n_commi': sum(a.get('n_commi', 1) for a in art_segs),
        'full_text': full_text,
        'slice_info': None,
        'errore': None,
    }


def _finalize_from_pdf(slug, label, pdf_path, meta, atto):
    full_fasc = estrai_testo_da_pdf(pdf_path)
    if not full_fasc:
        return _empty_result(slug, label, 'PDF vuoto o scansione')

    atto_text, status, info = slice_atto_from_fascicolo(full_fasc, atto)
    articoli  = estrai_articoli_da_testo(atto_text)
    cit_pdf   = estrai_citazioni_da_testo(atto_text)

    # unisci con citazioni URN da HTML (più affidabili perché già strutturate)
    seen = {c['urn'] for c in cit_pdf}
    all_cit = list(cit_pdf)
    for c in meta.get('urn_citazioni', []):
        if c['urn'] not in seen:
            seen.add(c['urn'])
            all_cit.append(c)

    return {
        'slug': slug, 'label': label,
        'fonte': f'pdf_gu_{status}',
        'fonte_file': pdf_path.name,
        'titolo_estratto': meta.get('titolo', '') or atto['titolo'],
        'preambolo': meta.get('preambolo', ''),
        'articoli': articoli,
        'citazioni': all_cit,
        'n_articoli': len(articoli),
        'n_citazioni': len(all_cit),
        'n_commi': sum(a.get('n_commi', 1) for a in articoli),
        'full_text': atto_text,
        'slice_info': info,
        'errore': None if articoli else 'nessun articolo dopo slicing',
    }


def process_atto(atto):
    """Pipeline completa per un atto.

    1. HTML Normattiva → meta + citazioni URN
    2. AKN: articoli strutturati → se >= MIN_ARTICOLI_AKN, usa questo
    3. Fallback: PDF GU → slicing per URN target → articoli
    4. Fallimento: ritorna struttura vuota con errore
    """
    slug, label = atto['slug'], atto['label']

    # 1. HTML Normattiva
    html, html_status = fetch_html_normattiva(atto)
    if not html:
        return _empty_result(slug, label, 'HTML Normattiva non disponibile')

    meta = estrai_meta_da_html(html)

    # 2. AKN HTML
    art_segs = estrai_articoli_multirichiesta(html, slug, atto['html_url'])
    if len(art_segs) >= MIN_ARTICOLI_AKN:
        res = _finalize_from_akn(slug, label, meta, art_segs, atto)
        print(f'  [{label:18s}] ✓ AKN     — {res["n_articoli"]:4d} art, {res["n_citazioni"]:3d} cit')
        return res

    # 3. Fallback PDF GU + slicing
    pdf_path, pdf_status = fetch_gu_pdf(meta.get('gu_pdf_url', ''), slug)
    if pdf_path:
        res = _finalize_from_pdf(slug, label, pdf_path, meta, atto)
        if res['errore']:
            print(f'  [{label:18s}] ⚠ PDF GU  — {res["errore"]}')
        else:
            info = res['slice_info'] or {}
            print(f'  [{label:18s}] ✂ PDF GU  — {res["n_articoli"]:4d} art, '
                  f'{res["n_citazioni"]:3d} cit  (atti nel fascicolo: {info.get("n_atti_unici", "?")})')
        return res

    # 4. Fallimento totale — almeno le citazioni HTML
    res = _empty_result(slug, label, 'AKN vuoto e PDF GU non disponibile')
    res['citazioni']   = meta.get('urn_citazioni', [])
    res['n_citazioni'] = len(res['citazioni'])
    res['titolo_estratto'] = meta.get('titolo', '') or atto['titolo']
    res['preambolo']       = meta.get('preambolo', '')
    print(f'  [{label:18s}] ✗ FAIL    — {res["errore"]}')
    return res


print('✓ Pipeline singolo atto: process_atto(atto)')

✓ Pipeline singolo atto: process_atto(atto)


In [6]:
# stesso test di prima
test_atto = next(a for a in ATTI_SEED if a['slug'] == 'l_136_2010')
print(f"Test: {test_atto['label']}")
print('=' * 60)

for p in RAW_DIR.glob(f'{test_atto["slug"]}_art_*.html'):
    p.unlink()

res = process_atto(test_atto)
print()
print(f'Fonte:           {res["fonte"]}')
print(f'n_articoli:      {res["n_articoli"]}  (atteso: ~7)')
print(f'titolo:          {res["titolo_estratto"][:80]}')
print(f'full_text chars: {len(res["full_text"])}')
print()
for art in res['articoli'][:3]:
    print(f'  Art. {art["numero"]} — rubrica: {art["rubrica"][:60]}')
    print(f'    testo (200 chars): {art["testo"][:200]}')
    print()

Test: L. 136/2010


C:\Users\claud\AppData\Local\Temp\ipykernel_24688\1095183128.py:305: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_fragment, 'html.parser')


  [L. 136/2010       ] ✓ AKN     —   16 art,  16 cit

Fonte:           html_akn
n_articoli:      16  (atteso: ~7)
titolo:          LEGGE



		



		 	



			13

			agosto

			2010, n. 136 Piano straordinario con
full_text chars: 53922

  Art. 1 — rubrica: (Delega al Governo per l'emanazione di un codice delle leggi
    testo (200 chars): Art. 1. (Delega al Governo per l'emanazione di un codice delle leggi antimafia e delle misure di prevenzione)
1. Il Governo è delegato ad adottare, senza nuovi o maggiori oneri per la finanza pubblica

  Art. 2 — rubrica: (Delega al Governo per l'emanazione di nuove disposizioni in
    testo (200 chars): Art. 2. (Delega al Governo per l'emanazione di nuove disposizioni in materia di documentazione antimafia)
1. Il Governo è delegato ad adottare, entro un anno dalla data di entrata in vigore della pres

  Art. 3 — rubrica: (Tracciabilità dei flussi finanziari)
    testo (200 chars): Art. 3. (Tracciabilità dei flussi finanziari)
1. Per assicurare la trac

## 6. Esecuzione Batch

Loop semplice. Nessuna `def` in questa cella — solo chiamate a `process_atto`.

In [ ]:
print(f'Processing {len(ATTI_SEED)} atti...')
print('=' * 78)

results = []
for i, atto in enumerate(ATTI_SEED):
    print(f'[{i+1:2d}/{len(ATTI_SEED)}]', end=' ')
    res = process_atto(atto)
    # arricchimento con metadati seed
    for k in ('tipo', 'data', 'numero', 'urn', 'titolo',
              'layer_atteso', 'note', 'eu_celex_collegati'):
        res[k] = atto.get(k, '')
    results.append(res)

print('=' * 78)
print(f'Completato: {len(results)} atti processati')

# Riepilogo per fonte
from collections import Counter
fonti = Counter(r['fonte'] for r in results)
print()
print('Fonti utilizzate:')
for k, v in fonti.most_common():
    print(f'  {k:25s}  {v:3d}')

Processing 29 atti...
[ 1/29] 

C:\Users\claud\AppData\Local\Temp\ipykernel_24688\1095183128.py:305: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_fragment, 'html.parser')


## 7. Validazione

Per ogni risultato verifica che il `full_text` sia davvero quello dell'atto
atteso. La validazione dipende dalla fonte:

- **AKN**: il testo non contiene l'heading ufficiale della GU ("DECRETO
  LEGISLATIVO..."), quindi si valida che il `titolo_estratto` dalla pagina
  Normattiva contenga il numero + anno dell'atto.
- **PDF GU sliced**: si controlla che la prima heading del `full_text` sia
  quella attesa.
- **Fallimento**: validazione fallisce automaticamente.

Se `HALT_ON_VALIDATION_FAIL = True`, un fallimento blocca il resto del notebook.

In [ ]:
def validate_result(res):
    """Ritorna (ok: bool, reason: str)."""
    if res['errore']:
        return False, f'errore pipeline: {res["errore"]}'
    if not res['full_text']:
        return False, 'full_text vuoto'

    atto_expected = (res['tipo'], res['data'], res['numero'])

    if res['fonte'] == 'html_akn':
        # AKN: valida che titolo/preambolo contenga numero+anno
        anno = res['data'][:4]
        num  = res['numero']
        hay = f'{res["titolo_estratto"]} {res["preambolo"][:500]}'
        # es. "n. 163" oppure "163/2006" oppure "163 del 2006"
        patterns = [
            rf'\bn\.?\s*{num}\b.*?\b{anno}\b',
            rf'\b{num}\s*/\s*{anno}\b',
            rf'\b{num}\s+del\s+{anno}\b',
        ]
        if any(re.search(p, hay, re.IGNORECASE | re.DOTALL) for p in patterns):
            return True, 'AKN: numero+anno nel titolo/preambolo'
        # se non c'è il matching numerico ma abbiamo articoli popolati,
        # consideriamo valido con warning soft (il titolo Normattiva può essere generico)
        if res['n_articoli'] >= 1 and res['titolo_estratto']:
            return True, 'AKN: articoli popolati (match titolo non verificabile)'
        return False, f'AKN: numero+anno non trovati nel titolo "{res["titolo_estratto"][:80]}"'

    if res['fonte'].startswith('pdf_gu'):
        heads = find_all_headings(res['full_text'])
        if not heads:
            return False, 'PDF slice: nessuna heading nel full_text'
        first = heads[0]
        if (first['tipo'], first['data'], first['numero']) == atto_expected:
            return True, f'PDF slice: prima heading = atteso ({first["data"]} n.{first["numero"]})'
        return False, (f'PDF slice: prima heading = {first["tipo"]} {first["data"]} '
                       f'n.{first["numero"]} (atteso {atto_expected[1]} n.{atto_expected[2]})')

    return False, f'fonte sconosciuta: {res["fonte"]}'


print('✓ validate_result definita')

In [ ]:
val_rows = []
for res in results:
    ok, reason = validate_result(res)
    val_rows.append({
        'label':        res['label'],
        'fonte':        res['fonte'],
        'n_articoli':   res['n_articoli'],
        'n_citazioni':  res['n_citazioni'],
        'validation':   '✓ OK' if ok else '✗ FAIL',
        'reason':       reason,
    })

df_val = pd.DataFrame(val_rows)
df_val.to_csv(OUT_DIR / 'validation_report.csv', index=False, encoding='utf-8')

n_ok   = (df_val['validation'] == '✓ OK').sum()
n_fail = (df_val['validation'] == '✗ FAIL').sum()

print('=' * 78)
print('VALIDAZIONE')
print('=' * 78)
print(f'  ✓ OK    {n_ok:3d}')
print(f'  ✗ FAIL  {n_fail:3d}')
print()
display(df_val)

if n_fail > 0 and HALT_ON_VALIDATION_FAIL:
    failures = df_val[df_val['validation'] == '✗ FAIL']
    raise RuntimeError(
        f'\nValidazione fallita per {n_fail} atti:\n'
        + '\n'.join(f'  - {r["label"]}: {r["reason"]}' for _, r in failures.iterrows())
        + '\n\nImposta HALT_ON_VALIDATION_FAIL=False per esportare comunque, '
          'oppure correggi la pipeline.'
    )

## 8. Quality Report

Score euristico per ogni atto (0-100) basato su numero articoli, citazioni,
lunghezza testo. Serve per il report finale e per il notebook 03 che pesa
gli atti per qualità.

In [ ]:
quality_rows = []
for res in results:
    n_art       = res.get('n_articoli', 0)
    n_cit       = res.get('n_citazioni', 0)
    testo_chars = sum(len(a.get('testo', '')) for a in res.get('articoli', []))
    errore      = res.get('errore')

    score = 0
    if not errore:             score += 20
    if n_art > 100:            score += 30
    elif n_art > 20:           score += 20
    elif n_art > 5:            score += 10
    elif n_art > 0:            score += 5
    if n_cit > 30:             score += 25
    elif n_cit > 10:           score += 15
    elif n_cit > 0:            score += 5
    if testo_chars > 100_000:  score += 25
    elif testo_chars > 20_000: score += 15
    elif testo_chars > 2_000:  score += 5

    quality_rows.append({
        'slug':          res['slug'],
        'label':         res['label'],
        'fonte':         res['fonte'],
        'n_articoli':    n_art,
        'n_citazioni':   n_cit,
        'testo_chars':   testo_chars,
        'quality_score': score,
        'errore':        errore or '',
    })

df_quality = pd.DataFrame(quality_rows).sort_values('quality_score', ascending=False)
print('QUALITY REPORT')
print('=' * 78)
display(df_quality)

## 9. Grafo Citazioni

Costruisce `edges_it.csv` con tutte le citazioni seed → qualsiasi atto,
e `edges_it_internal.csv` con le sole citazioni interne al catalogo.

In [ ]:
urn_to_slug = {a['urn']: a['slug'] for a in ATTI_SEED}


def normalize_urn(raw):
    m = re.search(r'(urn:nir:[^\s~&"<]+)', raw or '')
    return m.group(1).rstrip('/ ') if m else None


edges = []
for res in results:
    src_slug = res['slug']
    src_urn  = res['urn']
    for cit in res.get('citazioni', []):
        norm = normalize_urn(cit.get('urn', '')) or normalize_urn(cit.get('testo', ''))
        if not norm or norm == src_urn:
            continue
        edges.append({
            'src_slug':  src_slug,
            'src_urn':   src_urn,
            'dst_urn':   norm,
            'dst_slug':  urn_to_slug.get(norm),
            'testo_rif': cit.get('testo', '')[:200],
            'interno':   urn_to_slug.get(norm) is not None,
        })

df_edges = (pd.DataFrame(edges)
            .drop_duplicates(subset=['src_urn', 'dst_urn'])
            .reset_index(drop=True)) if edges else pd.DataFrame(
    columns=['src_slug', 'src_urn', 'dst_urn', 'dst_slug', 'testo_rif', 'interno'])

print(f'Citazioni estratte: {len(df_edges)}')
if len(df_edges) > 0:
    print(f'  di cui interne al catalogo: {df_edges["interno"].sum()}')
display(df_edges.head(20))

## 10. Export CSV

Scrive gli output finali compatibili con la pipeline (notebook 02→04):
`nodes_it.csv`, `nodes_texts_it.csv`, `edges_it.csv`, `edges_it_internal.csv`.

In [ ]:
nodes_rows = []
for res in results:
    slug  = res['slug']
    seed  = next(a for a in ATTI_SEED if a['slug'] == slug)
    score = next(r['quality_score'] for r in quality_rows if r['slug'] == slug)

    full_text = res.get('full_text', '')
    if not full_text and res.get('articoli'):
        # ricostruisci da articoli
        parts = [f'TITOLO: {res.get("titolo_estratto") or seed["titolo"]}']
        if res.get('preambolo'):
            parts.append(f'\nPREAMBOLO:\n{res["preambolo"]}')
        for art in res['articoli']:
            rubrica = f' — {art["rubrica"]}' if art.get('rubrica') else ''
            parts.append(f'\nArticolo {art["numero"]}{rubrica}\n{art["testo"]}')
        full_text = '\n'.join(parts)

    # ── Serializza articoli nel formato atteso da notebook 04 ─────────────────
    articoli = res.get('articoli', [])
    segments_json = json.dumps(
        [
            {
                'tipo':          'articolo',
                'identificatore': art.get('id', f'art{art.get("numero", i)}'),
                'numero':         art.get('numero', ''),
                'rubrica':        art.get('rubrica', ''),
                'testo':          art.get('testo', ''),
                'n_commi':        art.get('n_commi', 1),
            }
            for i, art in enumerate(articoli)
        ],
        ensure_ascii=False,
    )
    # text_status: ok se ha almeno un articolo, else 'no_text'
    text_status = 'ok' if articoli else 'no_text'

    nodes_rows.append({
        'Id':                 slug,      # alias atteso da notebook 04
        'Label':              slug,      # alias atteso da notebook 04
        'id':                 slug,
        'urn':                seed['urn'],
        'giurisdizione':      'IT',
        'tipo':               seed['tipo'],
        'anno':               int(seed['data'][:4]),
        'numero':             seed['numero'],
        'titolo':             res.get('titolo_estratto') or seed['titolo'],
        'layer_atteso':       seed['layer_atteso'],
        'eu_celex_collegati': json.dumps(seed.get('eu_celex_collegati', [])),
        'fonte_estrazione':   res.get('fonte', 'nessuna'),
        'fonte_file':         res.get('fonte_file', ''),
        'n_articoli':         res.get('n_articoli', 0),
        'n_commi':            res.get('n_commi', 0),
        'quality_score':      score,
        'note':               seed['note'],
        'text_status':        text_status,  # atteso da notebook 04
        'segments':           segments_json, # atteso da notebook 04
        'full_text':          full_text,
    })

df_nodes = pd.DataFrame(nodes_rows)

# nodes_it.csv — metadati (senza full_text)
nodes_path = OUT_DIR / 'nodes_it.csv'
df_nodes.drop(columns=['full_text']).to_csv(nodes_path, index=False, encoding='utf-8')
print(f'✓ {nodes_path}')

# nodes_texts_it.csv — con full_text per notebook 03/04
texts_path = OUT_DIR / 'nodes_texts_it.csv'
df_nodes.to_csv(texts_path, index=False, encoding='utf-8')
print(f'✓ {texts_path}  (con full_text)')

# edges_it.csv
edges_path = OUT_DIR / 'edges_it.csv'
df_edges.to_csv(edges_path, index=False, encoding='utf-8')
print(f'✓ {edges_path}  ({len(df_edges)} archi)')

# edges_it_internal.csv
if len(df_edges) > 0:
    df_int = df_edges[df_edges['interno']].copy()
    int_path = OUT_DIR / 'edges_it_internal.csv'
    df_int.to_csv(int_path, index=False, encoding='utf-8')
    print(f'✓ {int_path}  ({len(df_int)} archi interni)')

print()
print('Export completato.')

## 11. Integrazione EUR-Lex — Direttive europee sugli appalti

Estrazione delle direttive UE collegate (via campo `eu_celex_collegati` del
catalogo seed). Questa sezione è invariata rispetto alla versione precedente;
il codice completo è nel vecchio notebook `00_normattiva_explore.ipynb` alla
sezione 9. Se serve, si può copiare quella cella qui sotto as-is: non dipende
dalla nuova pipeline IT.

```python
# celex_ids = sorted({c for a in ATTI_SEED for c in a.get('eu_celex_collegati', [])})
# ... resto invariato ...
```